In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = (
    SparkSession.builder
        .appName("broze_app_ minio")
        .master("spark://spark-master:7077")

        # Config MinIO
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.access.key", "admin")
        .config("spark.hadoop.fs.s3a.secret.key", "admin123")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

        # Delta Lake
        .config("spark.sql.extensions",
                "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog")

        .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 02:16:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
raw_path = "s3a://analytics/raw/vendas/"

In [6]:
df_bronze = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(raw_path)
)

26/03/11 02:18:02 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [7]:
from pyspark.sql.functions import col

df_bronze = df_bronze.select([
    col(f"`{c}`").alias(c.replace(" ", "_").replace(".", "")) for c in df_bronze.columns
])


In [8]:
df_bronze.show(5)

+-------------------+--------------------+--------------+-------------+--------------+--------+-----------+-----------------+--------------------+
|      Data_da_Venda|             Produto|     Categoria|PrecoUnitario|Custo_Unitário|   Marca|Qtd_Vendida|     Nome_Cliente|          Localidade|
+-------------------+--------------------+--------------+-------------+--------------+--------+-----------+-----------------+--------------------+
|2017-06-01 00:00:00|Sistema de Som 7....|Sistema de Som|       1109.0|        367.43|Litware |        1.0|Pinheiro, Vicente|     França - Europa|
|2017-06-01 00:00:00|180 CFM Ventilado...|  Ventiladores|       215.62|         71.44| Litware|        1.0|    Lopez, Marlon|Estados Unidos - ...|
|2017-06-01 00:00:00|180 CFM Ventilado...|  Ventiladores|       215.62|         71.44| Litware|        1.0| Gonzalez, Martin|Chile - América d...|
|2017-06-01 00:00:00|180 CFM Ventilado...|  Ventiladores|       215.62|         71.44| Litware|        1.0|     Ruiz, 

In [9]:
df_bronze.printSchema()

root
 |-- Data_da_Venda: string (nullable = true)
 |-- Produto: string (nullable = true)
 |-- Categoria: string (nullable = true)
 |-- PrecoUnitario: double (nullable = true)
 |-- Custo_Unitário: double (nullable = true)
 |-- Marca: string (nullable = true)
 |-- Qtd_Vendida: double (nullable = true)
 |-- Nome_Cliente: string (nullable = true)
 |-- Localidade: string (nullable = true)



In [10]:
bronze_path = "s3a://analytics/medallion/01-bronze/vendas/"

In [11]:

(
    df_bronze.write.format("delta")
        .mode("overwrite")
        .save(bronze_path)
)

In [12]:
spark.stop()